# Mini-project — Gas liquefaction (LNG) 

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import caldis
from CoolProp.CoolProp import PropsSI

from caldis.core.system import System
from caldis.components import FluidBoundary
from caldis.components import Compressor
from caldis.components import ControlVolume
from caldis.components import SizingHX
from caldis.components import FlashTank
from caldis.components import Valve
from caldis.components import Splitter
from caldis.components import Mixer
from caldis.components import Turbine
from caldis.components.combustion.heater import SimpleHeater   # used here as a cooler
from caldis.fluids import CoolPropBackend
from caldis.solvers import solve                 # one-call workflow (DOF -> start -> solve)
from caldis.solvers.autostart import autostart, fix_port
from caldis.solvers.dynamic import simulate

In [3]:
# Constants
gas = "Methane"
methane_be = CoolPropBackend(gas)
T0 = 273.15 + 25.0  # K
P0 = 1e5 # Pa
P_high = 100e5
ic_T_out = T0 + 5 # K
comp_is_e = 0.75 
HX_pinch = 5 # K
HP_recup_coeff = 1000 # W/m2K
LP_recup_coeff = 500 # W/m2K
comp_r = 3.16 
mdot_liq = 1 # kg/s

In [ ]:
# Components

meth_inlet = FluidBoundary(name="Methane_inlet", backend=methane_be, T=T0, p=P0)
mix_in = Mixer(name="Mixer_in", backend=methane_be)
comp_1 = Compressor(name="Compressor_1", backend=methane_be, eta_is=comp_is_e)
comp_2 = Compressor(name="Compressor_2", backend=methane_be, eta_is=comp_is_e)
comp_3 = Compressor(name="Compressor_3", backend=methane_be, eta_is=comp_is_e)
comp_4 = Compressor(name="Compressor_4", backend=methane_be, eta_is=comp_is_e)
ic_1 = SimpleHeater(name="Intercooler_1", backend=methane_be)
ic_2 = SimpleHeater(name="Intercooler_2", backend=methane_be)
ic_3 = SimpleHeater(name="Intercooler_3", backend=methane_be)
ac = SimpleHeater(name="Aftercooler", backend=methane_be)
splitter = Splitter(name="Splitter", backend=methane_be)
expander = Turbine(name="Expander", backend=methane_be, eta_is=0.8)
recup = SizingHX(name="Recuperator", backend_hot=methane_be, backend_cold=methane_be, h_hot=HP_recup_coeff, h_cold=LP_recup_coeff, area_ratio_cold_to_hot=1.0)
valve = Valve(name="Valve", backend=methane_be)
mix_out = Mixer(name="Mixer_out", backend=methane_be)
flash = FlashTank(name="Flash", backend=methane_be)
meth_outlet = FluidBoundary(name="Methane_outlet", backend=methane_be, mdot=mdot_liq)

In [34]:
mix_in.structure()

Mixer 'Mixer_in'
  port   in1  (Methane)   -> .p .mdot .h
  port   in2  (Methane)   -> .p .mdot .h
  port   out  (Methane)   -> .p .mdot .h


In [35]:
#System definition 
sys = System(name="LNG_L0")

sys.add(mix_in, comp_1, ic_1, comp_2, ic_2, comp_3, ic_3, comp_4, ac, splitter, expander, recup, valve, mix_out, flash, meth_inlet, meth_outlet)
sys.connect(meth_inlet.port, mix_in.in1)
sys.connect(mix_in.out, comp_1.inl)
sys.connect(comp_1.out, ic_1.inl)
sys.connect(ic_1.out, comp_2.inl)
sys.connect(comp_2.out, ic_2.inl)
sys.connect(ic_2.out, comp_3.inl)
sys.connect(comp_3.out, ic_3.inl)
sys.connect(ic_3.out, comp_4.inl)
sys.connect(comp_4.out, ac.inl)
sys.connect(ac.out, splitter.inl)

# splitter's 1-x_e
sys.connect(splitter.out1, recup.hot_in)
sys.connect(recup.hot_out, valve.inl)
sys.connect(valve.out, mix_out.in1)
##
# splitter's x_e
sys.connect(splitter.out2, expander.inl)
sys.connect(expander.out, mix_out.in2)
##

sys.connect(mix_out.out, flash.inl)

sys.connect(flash.liq, meth_outlet.port)
sys.connect(flash.vap, recup.cold_in)
sys.connect(recup.cold_out, mix_in.in2)

sys.add_constraint(lambda : comp_1.out.p - comp_r*P0, name="comp_1_pressure_constraint")
sys.add_constraint(lambda : comp_2.out.p - (comp_r**2)*P0, name="comp_2_pressure_constraint")
sys.add_constraint(lambda : comp_3.out.p - (comp_r**3)*P0, name="comp_3_pressure_constraint")
sys.add_constraint(lambda : comp_4.out.p - (comp_r**4)*P0, name="comp_4_pressure_constraint")
sys.add_constraint(lambda : ic_1.out.T - ic_T_out, name="ic_1_temperature_constraint")
sys.add_constraint(lambda : ic_2.out.T - ic_T_out, name="ic_2_temperature_constraint")
sys.add_constraint(lambda : ic_3.out.T - ic_T_out, name="ic_3_temperature_constraint")
sys.add_constraint(lambda : ac.out.T - ic_T_out, name="ac_temperature_constraint")
sys.add_constraint(lambda : flash.inl.p - 1e5, name="flash_pressure_constraint")
sys.add_constraint(lambda: recup.hot_in.T - recup.cold_out.T - HX_pinch, name="recup_warm_end_pinch")


<function __main__.<lambda>()>

In [ ]:
from caldis.solvers.autostart import autostart

pk = [1e5 * 3.16**k for k in range(5)]          # 1 ; 3,16 ; 9,99 ; 31,6 ; 99,7 bar
T_comp_out = [411.6, 417.9, 418.6, 420.4]       # K
comps, coolers = [comp_1, comp_2, comp_3, comp_4], [ic_1, ic_2, ic_3, ac]

for k, (C, X) in enumerate(zip(comps, coolers)):
    sys.set_start_port(C.inl,  p=pk[k],   T=T0 if k == 0 else ic_T_out)
    sys.set_start_port(C.out,  p=pk[k+1], T=T_comp_out[k])
    sys.set_start_port(X.inl,  p=pk[k+1], T=T_comp_out[k])
    sys.set_start_port(X.out,  p=pk[k+1], T=ic_T_out)

for prt in (meth_inlet.port, mix_in.in1, mix_in.in2, mix_in.out):
    sys.set_start_port(prt, p=1e5, T=T0)

sys.set_start_port(recup.hot_in, p=pk[4], T=ic_T_out)
sys.set_start_port(recup.hot_out, p=pk[4], T=217.0)
sys.set_start_port(valve.inl, p=pk[4], T=217.0)
sys.set_start_port(recup.cold_in, p=1e5, T=113.0)  # vapeur sat., légèrement surchauffée
sys.set_start_port(flash.vap, p=1e5, T=113.0)
sys.set_start_port(recup.cold_out, p=1e5, T=T0)
sys.set_start_port(flash.liq, p=1e5, T=110.0)  # liquide sat., légèrement sous-refroidi
sys.set_start_port(recup.hot_out, h=PropsSI("H", "T", 217.0, "P", pk[4], gas))
sys.set_start_port(mix_out.in1, h=PropsSI("H", "T", T0, "P", 1e5, gas))
sys.set_start_port(valve.inl, h=PropsSI("H", "T", 217.0, "P", pk[4], gas))
sys.set_start_port(valve.out, h=PropsSI("H", "T", 113.0, "P", 1e5, gas))


autostart(sys)
sol = solve(sys, verbose=True)

✓ DOF: square (121 unknowns)
✓ start point: evaluable
⚠ 7 start-sanity warning(s):
   - Mixer_in: out mdot should be < 0 (outlet)
   - Splitter: out1 mdot should be < 0 (outlet)
   - Splitter: out2 mdot should be < 0 (outlet)
   - Expander: p_out (1.00 bar) must be < p_in (1.00 bar) for expansion
   - Expander: inlet 283 K seems cold for a gas turbine (forgotten hot start?)
   - Mixer_out: out mdot should be < 0 (outlet)
   - Flash: vap mdot should be < 0 (outlet)
  (non-blocking.)
  direct Newton: failed -> fallback to homotopy (1 it, |F_scaled|=9.9e-01)
✗ FAILED: homotopy stuck at lambda=0.0000 (dlam<0.001): jacobienne singulière (it 1)

=== convergence report (FAILED) ===
message: homotopy stuck at lambda=0.0000 (dlam<0.001): jacobienne singulière (it 1)
|R_scaled|inf = 9.900e-01   (|R_raw|inf = 9.871e+06)

Top 8 violated equations (by SCALED residual):
  scaled= 9.90e-01  (raw=-9.87e+06)   Splitter.p_out2
  scaled= 9.80e-01  (raw=-9.80e-01)   Flash.mass
  scaled= 6.67e-01  (raw=-2.

In [48]:
solve(sys)

✓ DOF: square (121 unknowns)
✓ start point: evaluable
⚠ 15 start-sanity warning(s):
   - Mixer_in: out mdot should be < 0 (outlet)
   - Compressor_1: p_out (1.00 bar) must be > p_in (1.00 bar) for compression
   - Compressor_2: p_out (1.00 bar) must be > p_in (1.00 bar) for compression
   - Compressor_3: p_out (1.00 bar) must be > p_in (1.00 bar) for compression
   - Compressor_4: p_out (1.00 bar) must be > p_in (1.00 bar) for compression
   - Splitter: out1 mdot should be < 0 (outlet)
   - Splitter: out2 mdot should be < 0 (outlet)
   - Expander: p_out (1.00 bar) must be < p_in (1.00 bar) for expansion
   - Expander: inlet 112 K seems cold for a gas turbine (forgotten hot start?)
   - Recuperator: temperature cross at start (dT1=0.0, dT2=0.0 K)
   - Recuperator: hot inlet/outlet nearly equal -> LMTD/flow degenerate
   - Recuperator: cold inlet/outlet nearly equal -> cold MASS FLOW is undetermined (anchor the cold-side inlet/outlet TEMPERATURES)
   - Mixer_out: out mdot should be < 0 (

 message: homotopy stuck at lambda=0.0000 (dlam<0.001): jacobienne singulière (it 1)
 success: False
     fun: [ 3.906e-05  1.172e+01 ...  0.000e+00 -6.510e-03]
       x: [ 1.000e+05  1.000e-02 ...  1.000e+00  3.000e+05]
     nit: 9

In [46]:
def warm_start(sys_):
    for v in sys_.unknowns:
        v.start = v.value # the current solution becomes the starting point
        
        # after a successful solve at x=0.1 (seeds + autostart already done):
for xe in [0.01, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5]:
    warm_start(sys) # restart from the previous solution
    x_expander = xe # the split-constraint lambda reads this global
    sol = solve(sys, steps=40)
    print(f"x_e={xe:.2f} success={sol.success} m_circ={mix_out.out.mdot:.2f}")
    if not sol.success:
        break

✓ DOF: square (121 unknowns)
✓ start point: evaluable
⚠ 15 start-sanity warning(s):
   - Mixer_in: out mdot should be < 0 (outlet)
   - Compressor_1: p_out (1.00 bar) must be > p_in (1.00 bar) for compression
   - Compressor_2: p_out (1.00 bar) must be > p_in (1.00 bar) for compression
   - Compressor_3: p_out (1.00 bar) must be > p_in (1.00 bar) for compression
   - Compressor_4: p_out (1.00 bar) must be > p_in (1.00 bar) for compression
   - Splitter: out1 mdot should be < 0 (outlet)
   - Splitter: out2 mdot should be < 0 (outlet)
   - Expander: p_out (1.00 bar) must be < p_in (1.00 bar) for expansion
   - Expander: inlet 112 K seems cold for a gas turbine (forgotten hot start?)
   - Recuperator: temperature cross at start (dT1=0.0, dT2=0.0 K)
   - Recuperator: hot inlet/outlet nearly equal -> LMTD/flow degenerate
   - Recuperator: cold inlet/outlet nearly equal -> cold MASS FLOW is undetermined (anchor the cold-side inlet/outlet TEMPERATURES)
   - Mixer_out: out mdot should be < 0 (

RecursionError: maximum recursion depth exceeded